# 09. Denoising and Restoration Without Inventing Biology

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook explains restoration as an inverse/statistical problem and emphasizes scientific validation over visual attractiveness.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Distinguish paired restoration from generic image enhancement
- Understand modality-specific noise assumptions
- Compare L1/L2/PSNR/SSIM
- Design hallucination stress tests
- Validate OCT and fluorescence restoration scientifically


## Mind map

```mermaid
mindmap
  root((Restoration))
    Data
      Paired
      Simulated degradation
      Real noise
    Loss
      L1
      MSE
      Structural
    Metrics
      PSNR
      SSIM
      Task metrics
    Risks
      Oversmoothing
      Hallucination
      Domain shift
    Modalities
      OCT
      Fluorescence

```


## 1. Restoration is not the same as segmentation

In restoration, every output pixel is an estimate of an underlying image. The model can make the image look plausible while being scientifically wrong.

Core question:

> What evidence tells me that a restored structure was supported by the measurement rather than invented by the network?


## 2. Supervised paired restoration

You need aligned pairs:

```text
noisy / low-quality input  ->  clean / high-quality target
```

Possible ways to obtain pairs:
- repeated acquisitions + averaging;
- low/high exposure;
- low/high photon budget;
- simulated degradation from high-quality data;
- registered modalities.

Each pairing strategy has assumptions. A simulated degradation may not represent real instrument noise.


## 3. Noise models matter

Fluorescence often contains:
- Poisson-like shot noise;
- read noise;
- dark current/background;
- detector-specific effects.

OCT speckle is not simply additive Gaussian noise. It is linked to coherent interference and tissue microstructure. A denoising model trained with the wrong synthetic noise may learn the wrong mapping.


## 4. L1 versus L2

```text
L1  = mean(|prediction-target|)
L2  = mean((prediction-target)^2)
```

L2 strongly penalizes large errors and can encourage averaged/smoothed outputs. L1 is often sharper but neither loss alone guarantees preservation of biologically meaningful fine structures.


In [ ]:
import torch
import torch.nn.functional as F

target = torch.tensor([0.,0.,1.,1.])
pred1 = torch.tensor([0.,0.,0.7,1.3])
pred2 = torch.tensor([0.,0.,0.9,2.0])

for name,pred in [("pred1",pred1),("pred2",pred2)]:
    print(name,
          "L1=",F.l1_loss(pred,target).item(),
          "MSE=",F.mse_loss(pred,target).item())


## 5. PSNR

For maximum signal value `MAX` and MSE:

```text
PSNR = 10 * log10(MAX^2 / MSE)
```

PSNR is useful when intensity scaling is defined and the target is trustworthy.

### Caution
A higher PSNR does not guarantee better preservation of small structures or diagnostic features.


## 6. SSIM

SSIM compares local luminance, contrast, and structure. It can be more perceptually aligned than MSE, but it still cannot prove that a structure is real.

Use SSIM as one piece of evidence, not biological ground truth.


## 7. Hallucination stress tests

For optical restoration, test:
- dim isolated structures;
- empty/background regions;
- edge-of-field artifacts;
- unusual morphology;
- low SNR beyond training range;
- acquisition settings not seen in training;
- intentionally inserted synthetic structures with known truth;
- known resolution targets/phantoms when available.

Compare input, target, output, and error map.


## 8. OCT-specific validation ideas

Depending on the scientific question:
- layer boundary preservation;
- attenuation/intensity profile preservation;
- speckle statistics;
- CNR/SNR in defined ROIs;
- resolution/edge spread;
- downstream segmentation stability;
- repeatability across repeated scans.

If quantitative OCT intensity is used downstream, verify that the restoration does not alter the quantitative relationship you care about.


## 9. Fluorescence-specific validation ideas

- detectability of dim puncta;
- integrated intensity conservation if scientifically required;
- object count stability;
- line-profile width/resolution;
- bleaching/exposure domain shift;
- low-photon regime performance;
- comparison against classical denoising baseline.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
